In [4]:
import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent

train_df = pd.read_csv(
    project_dir / "single_family_train_cleaned.csv",
    low_memory=False
)

test_df = pd.read_csv(
    project_dir / "single_family_test_cleaned.csv",
    low_memory=False
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nAvailable columns:")
print(train_df.columns.tolist())

Training shape: (278710, 26)
Test shape: (11684, 26)

Available columns:
['Flooring', 'ViewYN', 'PoolPrivateYN', 'Latitude', 'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'SubdivisionName', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN', 'Stories', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'LotSizeSquareFeet', 'ClosePrice', 'CloseDate']


In [7]:
feature_columns = [
    "YearBuilt",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LivingArea",
    "LotSizeSquareFeet",
    "Latitude",
    "Longitude"
]

display(train_df[feature_columns].describe())

print("\nMissing values:")
display(train_df[feature_columns + ["CloseDate"]].isna().sum())

train_close_dates = pd.to_datetime(
    train_df["CloseDate"], errors="coerce", utc=True
)

print("\nTraining close date range:")
print("Earliest:", train_close_dates.min())
print("Latest:", train_close_dates.max())
print("Missing or invalid dates:", train_close_dates.isna().sum())

print(
    "\nRows with zero bathrooms:",
    (train_df["BathroomsTotalInteger"] == 0).sum()
)

,YearBuilt,BedroomsTotal,BathroomsTotalInteger,LivingArea,LotSizeSquareFeet,Latitude,Longitude
count,278710.000000,278710.000000,278710.000000,278710.000000,278710.000000,278710.000000,278710.000000
mean,1975.459309,3.464163,2.554002,1972.863633,12365.644895,34.801546,-118.713225
std,27.244585,0.866133,0.953381,811.662752,22411.680952,1.718562,1.896435
min,1776.000000,0.000000,0.000000,734.000000,1827.000000,32.122560,-124.193201
25%,1956.000000,3.000000,2.000000,1380.000000,5680.000000,33.769164,-119.769889
50%,1975.000000,3.000000,2.000000,1795.000000,7200.000000,34.093175,-118.046746
75%,1997.000000,4.000000,3.000000,2382.000000,9871.000000,35.299792,-117.268705
max,2026.000000,6.000000,6.000000,5648.000000,252212.000000,41.912794,-114.311408



Missing values:


YearBuilt                0
BedroomsTotal            0
BathroomsTotalInteger    0
LivingArea               0
LotSizeSquareFeet        0
Latitude                 0
Longitude                0
CloseDate                0
dtype: int64


Training close date range:
Earliest: 2024-01-01 00:00:00+00:00
Latest: 2026-03-31 00:00:00+00:00
Missing or invalid dates: 0

Rows with zero bathrooms: 55


In [8]:
# Create independent copies for feature engineering
train_fe = train_df.copy()
test_fe = test_df.copy()

for data in [train_fe, test_fe]:
    close_year = pd.to_datetime(
        data["CloseDate"], errors="coerce", utc=True
    ).dt.year

    data["PropertyAge"] = close_year - data["YearBuilt"]

    data["BedBathRatio"] = (
        data["BedroomsTotal"]
        / data["BathroomsTotalInteger"].replace(0, float("nan"))
    )

# Inspect the engineered features
for name, data in [("Training", train_fe), ("Test", test_fe)]:
    print(f"\n{name} dataset:")
    display(data[["PropertyAge", "BedBathRatio"]].describe())

    print("Negative property ages:", (data["PropertyAge"] < 0).sum())
    print("Missing bed/bath ratios:", data["BedBathRatio"].isna().sum())


Training dataset:


,PropertyAge,BedBathRatio
count,278710.000000,278655.000000
mean,49.165922,1.474486
std,27.243497,0.471898
min,0.000000,0.000000
25%,27.000000,1.000000
50%,49.000000,1.500000
75%,69.000000,1.666667
max,249.000000,6.000000


Negative property ages: 0
Missing bed/bath ratios: 55

Test dataset:


,PropertyAge,BedBathRatio
count,11684.000000,11682.000000
mean,50.050839,1.460406
std,27.237527,0.468798
min,0.000000,0.500000
25%,28.000000,1.000000
50%,50.000000,1.500000
75%,70.000000,1.500000
max,146.000000,5.000000


Negative property ages: 0
Missing bed/bath ratios: 2


In [9]:
# Recalculate ratios so this cell is safe to rerun
for data in [train_fe, test_fe]:
    data["ZeroBathrooms"] = (
        data["BathroomsTotalInteger"] == 0
    ).astype(int)

    data["BedBathRatio"] = (
        data["BedroomsTotal"]
        / data["BathroomsTotalInteger"].replace(0, float("nan"))
    )

# Use the training median for both datasets
ratio_median = train_fe["BedBathRatio"].median()

for name, data in [("Training", train_fe), ("Test", test_fe)]:
    data["BedBathRatio"] = data["BedBathRatio"].fillna(ratio_median)

    print(f"\n{name} dataset:")
    print("Zero-bathroom records:", data["ZeroBathrooms"].sum())
    print("Missing ratios:", data["BedBathRatio"].isna().sum())

print("\nMedian used:", ratio_median)


Training dataset:
Zero-bathroom records: 55
Missing ratios: 0

Test dataset:
Zero-bathroom records: 2
Missing ratios: 0

Median used: 1.5



### School District Geographic Features

To add more detailed geographic information, we will spatially join each property's latitude and longitude with the California School District Areas 2024–25 boundaries. The resulting school district features may capture local differences in property prices beyond city or ZIP code.

First, we check whether GeoPandas is installed and whether school district boundary files are available in the project folder.

In [11]:
%pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 6.3 MB/s  0:00:03m 6.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 5.0 MB/s  0:00:00m 5.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 5.9 MB/s  0:00:006.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]━ 3/4 [geopandas]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path
import geopandas as gpd

project_dir = Path.cwd().parent

district_path = (
    project_dir
    / "DistrictAreas2425"
    / "DistrictAreas2425.shp"
)

districts = gpd.read_file(district_path)

print("Number of districts:", len(districts))
print("Coordinate system:", districts.crs)
print("\nAvailable columns:")
print(districts.columns.tolist())

display(districts.head())

Number of districts: 937
Coordinate system: EPSG:3857

Available columns:
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe', 'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len', 'geometry']


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [19]:
# Convert district boundaries to latitude/longitude coordinates
districts = districts.to_crs(epsg=4326)

# Inspect district types before spatial matching
print("District types:")
print(districts["DistrictTy"].value_counts(dropna=False))

display(
    districts[
        ["DistrictNa", "DistrictTy", "CDSCode"]
    ].head(10)
)

District types:
DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: int64


,DistrictNa,DistrictTy,CDSCode
0,Alameda Unified,Unified,01611190000000
1,Albany City Unified,Unified,01611270000000
2,Berkeley Unified,Unified,01611430000000
3,Castro Valley Unified,Unified,01611500000000
4,Emery Unified,Unified,01611680000000
5,Fremont Unified,Unified,01611760000000
6,Hayward Unified,Unified,01611920000000
7,Livermore Valley Joint Unified,Unified,01612000000000
8,Mountain House Elementary,Elementary,01612180000000
9,Newark Unified,Unified,01612340000000


In [20]:
# Select districts serving high school grades
high_school_districts = districts.loc[
    districts["DistrictTy"].isin(["High", "Unified"]),
    ["CDSCode", "DistrictNa", "DistrictTy", "geometry"]
].copy()

print("Districts selected:", len(high_school_districts))

Districts selected: 421


In [21]:
def match_school_districts(data):
    # Create geographic points from property coordinates
    points = gpd.GeoDataFrame(
        data[["Latitude", "Longitude"]].copy(),
        geometry=gpd.points_from_xy(
            data["Longitude"], data["Latitude"]
        ),
        crs="EPSG:4326"
    ).to_crs(high_school_districts.crs)

    # Find the district containing each property
    return gpd.sjoin(
        points,
        high_school_districts,
        how="left",
        predicate="within"
    )


train_school_matches = match_school_districts(train_fe)
test_school_matches = match_school_districts(test_fe)

# Check the number of matches for each property record
for name, matches in [
    ("Training", train_school_matches),
    ("Test", test_school_matches)
]:
    match_counts = (
        matches["index_right"].notna()
        .groupby(level=0).sum()
    )

    print(f"\n{name} dataset:")
    print("Properties matched once:", (match_counts == 1).sum())
    print("Unmatched properties:", (match_counts == 0).sum())
    print("Properties with multiple matches:", (match_counts > 1).sum())


Training dataset:
Properties matched once: 278510
Unmatched properties: 200
Properties with multiple matches: 0

Test dataset:
Properties matched once: 11673
Unmatched properties: 11
Properties with multiple matches: 0


In [24]:
# Inspect properties without a spatial district match
for name, data, matches in [
    ("Training", train_fe, train_school_matches),
    ("Test", test_fe, test_school_matches)
]:
    unmatched_index = matches.index[
        matches["index_right"].isna()
    ]

    print(f"\n{name}: unmatched property examples")
    display(
        data.loc[
            unmatched_index,
            ["Latitude", "Longitude", "HighSchoolDistrict"]
        ].head(15)
    )


Training: unmatched property examples


,Latitude,Longitude,HighSchoolDistrict
973,33.867826,-118.388314,hermosa
4071,33.864909,-118.401071,unknown
4468,38.701410,-122.375039,napa valley unified
6617,33.856680,-118.386208,hermosa
6629,33.872163,-118.398730,manhattan unified
6999,33.855277,-118.391960,manhattan unified
7849,33.865812,-118.398681,hermosa
11178,33.857305,-118.385768,manhattan unified
14166,33.872103,-118.401348,hermosa
14891,33.873392,-118.405380,hermosa



Test: unmatched property examples


,Latitude,Longitude,HighSchoolDistrict
75,33.872092,-118.400431,manhattan unified
185,33.870898,-118.404933,manhattan unified
1195,33.866297,-118.387827,hermosa
1806,33.868983,-118.401717,manhattan unified
2281,33.857141,-118.388476,manhattan unified
5688,33.856916,-118.386124,manhattan unified
7350,33.874962,-118.400283,hermosa
7434,33.865437,-118.388504,hermosa
9348,33.868871,-118.397126,manhattan unified
9416,33.871802,-118.404743,hermosa


In [25]:
# Check unmatched properties against all district types
for name, matches in [
    ("Training", train_school_matches),
    ("Test", test_school_matches)
]:
    unmatched_points = matches.loc[
        matches["index_right"].isna(),
        ["Latitude", "Longitude", "geometry"]
    ].copy()

    all_district_matches = gpd.sjoin(
        unmatched_points.to_crs(districts.crs),
        districts[
            ["CDSCode", "DistrictNa", "DistrictTy", "geometry"]
        ],
        how="left",
        predicate="within"
    )

    print(f"\n{name}: district matches using all types")
    display(
        all_district_matches[
            ["DistrictNa", "DistrictTy"]
        ].value_counts(dropna=False).rename("Match count").to_frame()
    )


Training: district matches using all types


,,Match count
DistrictNa,DistrictTy,
Hermosa Beach City Elementary,Elementary,169
NaN,NaN,29
Pope Valley Union Elementary,Elementary,1
Howell Mountain Elementary,Elementary,1



Test: district matches using all types


,,Match count
DistrictNa,DistrictTy,
Hermosa Beach City Elementary,Elementary,11


In [26]:
# Recheck unmatched training properties against all districts
unmatched_points = train_school_matches.loc[
    train_school_matches["index_right"].isna(),
    ["Latitude", "Longitude", "geometry"]
].copy()

train_all_matches = gpd.sjoin(
    unmatched_points.to_crs(districts.crs),
    districts[["DistrictNa", "DistrictTy", "geometry"]],
    how="left",
    predicate="within"
)

# Identify properties with no match across any district type
remaining_index = train_all_matches.index[
    train_all_matches["index_right"].isna()
].unique()

columns = [
    col for col in [
        "Latitude", "Longitude", "City",
        "PostalCode", "HighSchoolDistrict"
    ]
    if col in train_fe.columns
]

print("Properties still unmatched:", len(remaining_index))
display(train_fe.loc[remaining_index, columns])

Properties still unmatched: 29


,Latitude,Longitude,City,PostalCode,HighSchoolDistrict
31316,37.567119,-122.252418,foster city,94404,other
50763,37.908103,-122.380491,richmond,94801,other
57727,37.568311,-122.255087,foster city,94404,other
85083,37.764629,-122.226039,alameda,94501,unknown
88630,32.628150,-117.132141,coronado,92118,unknown
88774,37.874945,-122.473611,belvedere,94920,unknown
92961,37.570844,-122.258970,foster city,94404,other
110153,37.762060,-122.224984,alameda,94501,alameda city unified
118414,32.629325,-117.133681,coronado,92118,unknown
124103,37.909510,-122.380323,richmond,94801,unknown


In [27]:
# Select districts serving elementary grades
elementary_districts = districts.loc[
    districts["DistrictTy"].isin(["Elementary", "Unified"]),
    ["CDSCode", "DistrictNa", "DistrictTy", "geometry"]
].copy()


def match_elementary_districts(school_matches):
    # Reuse all property coordinates from the previous matching step
    points = school_matches[
        ["Latitude", "Longitude", "geometry"]
    ].copy().to_crs(elementary_districts.crs)

    return gpd.sjoin(
        points,
        elementary_districts,
        how="left",
        predicate="within"
    )


train_elementary_matches = match_elementary_districts(
    train_school_matches
)
test_elementary_matches = match_elementary_districts(
    test_school_matches
)

# Check unmatched properties and multiple matches
for name, matches in [
    ("Training", train_elementary_matches),
    ("Test", test_elementary_matches)
]:
    match_counts = (
        matches["index_right"].notna()
        .groupby(level=0).sum()
    )

    print(f"\n{name} dataset:")
    print("Properties matched once:", (match_counts == 1).sum())
    print("Unmatched properties:", (match_counts == 0).sum())
    print("Properties with multiple matches:", (match_counts > 1).sum())


Training dataset:
Properties matched once: 277539
Unmatched properties: 29
Properties with multiple matches: 1142

Test dataset:
Properties matched once: 11638
Unmatched properties: 0
Properties with multiple matches: 46


Multiple matches indicate that a property's coordinates fall within more than one selected elementary-level school district boundary.

In [29]:
# Inspect properties matching multiple elementary-level districts
for name, matches in [
    ("Training", train_elementary_matches),
    ("Test", test_elementary_matches)
]:
    match_counts = (
        matches["index_right"].notna()
        .groupby(level=0).sum()
    )

    multiple_index = match_counts[
        match_counts > 1
    ].index[:5]

    print(f"\n{name}: multiple-match examples")
    display(
        matches.loc[
            multiple_index,
            [
                "Latitude", "Longitude",
                "CDSCode", "DistrictNa", "DistrictTy"
            ]
        ].sort_index()
    )


Training: multiple-match examples


,Latitude,Longitude,CDSCode,DistrictNa,DistrictTy
134,34.422741,-119.627983,42692520000000,Montecito Union Elementary,Elementary
134,34.422741,-119.627983,42767860000000,Santa Barbara Unified,Unified
893,34.075392,-118.093932,19645500000000,Garvey Elementary,Elementary
893,34.075392,-118.093932,19757130000000,Alhambra Unified,Unified
1179,38.649317,-121.455416,34674210000000,Robla Elementary,Elementary
1179,38.649317,-121.455416,34765050000000,Twin Rivers Unified,Unified
1532,34.380273,-119.066496,56725040000000,Mupu Elementary,Elementary
1532,34.380273,-119.066496,56768280000000,Santa Paula Unified,Unified
1682,35.719822,-120.878586,40688250000000,San Miguel Joint Union,Elementary
1682,35.719822,-120.878586,40754570000000,Paso Robles Joint Unified,Unified



Test: multiple-match examples


,Latitude,Longitude,CDSCode,DistrictNa,DistrictTy
382,38.649785,-121.466837,34674210000000,Robla Elementary,Elementary
382,38.649785,-121.466837,34765050000000,Twin Rivers Unified,Unified
450,37.708386,-121.450366,39685440000000,Jefferson Elementary,Elementary
450,37.708386,-121.450366,39754990000000,Tracy Joint Unified,Unified
783,37.341342,-119.700218,20651850000000,Bass Lake Joint Union Elementary,Elementary
783,37.341342,-119.700218,20764140000000,Yosemite Unified,Unified
941,37.699114,-121.423612,39685440000000,Jefferson Elementary,Elementary
941,37.699114,-121.423612,39754990000000,Tracy Joint Unified,Unified
1891,38.656086,-121.461816,34674210000000,Robla Elementary,Elementary
1891,38.656086,-121.461816,34765050000000,Twin Rivers Unified,Unified


In [30]:
# Summarize district-type combinations for multiple matches
for name, matches in [
    ("Training", train_elementary_matches),
    ("Test", test_elementary_matches)
]:
    matched = matches.loc[matches["index_right"].notna()]

    type_combinations = matched.groupby(level=0)["DistrictTy"].agg(
        lambda types: " + ".join(sorted(types))
    )

    match_counts = matched.groupby(level=0).size()

    print(f"\n{name}: multiple-match type combinations")
    display(
        type_combinations.loc[match_counts > 1]
        .value_counts()
        .rename("Property count")
        .to_frame()
    )


Training: multiple-match type combinations


,Property count
DistrictTy,
Elementary + Unified,1142



Test: multiple-match type combinations


,Property count
DistrictTy,
Elementary + Unified,46


In [31]:
# Prefer Elementary when both Elementary and Unified match
def select_elementary_match(matches):
    selected = matches.copy()

    selected["_priority"] = selected["DistrictTy"].map(
        {"Elementary": 0, "Unified": 1}
    ).fillna(2)

    selected = selected.sort_values(
        "_priority", kind="stable"
    )

    selected = selected.loc[
        ~selected.index.duplicated(keep="first")
    ]

    return selected.drop(columns="_priority").sort_index()


train_elementary_selected = select_elementary_match(
    train_elementary_matches
)
test_elementary_selected = select_elementary_match(
    test_elementary_matches
)

# Verify one row per property
for name, selected in [
    ("Training", train_elementary_selected),
    ("Test", test_elementary_selected)
]:
    print(f"\n{name} dataset:")
    print("Property records:", len(selected))
    print("Duplicate indices:", selected.index.duplicated().sum())
    print("Unmatched properties:", selected["index_right"].isna().sum())


Training dataset:
Property records: 278710
Duplicate indices: 0
Unmatched properties: 29

Test dataset:
Property records: 11684
Duplicate indices: 0
Unmatched properties: 0


In [32]:
# Add spatial district features to the property datasets
for data, high_matches, elementary_matches in [
    (train_fe, train_school_matches, train_elementary_selected),
    (test_fe, test_school_matches, test_elementary_selected)
]:
    assert data.index.is_unique
    assert high_matches.index.is_unique
    assert elementary_matches.index.is_unique
    assert data.index.equals(high_matches.reindex(data.index).index)

    for prefix, matches in [
        ("SpatialHigh", high_matches),
        ("SpatialElementary", elementary_matches)
    ]:
        data[f"{prefix}DistrictCode"] = matches["CDSCode"].astype("string")
        data[f"{prefix}DistrictName"] = matches["DistrictNa"]
        data[f"{prefix}DistrictType"] = matches["DistrictTy"]

# Verify the added features
district_columns = [
    "SpatialHighDistrictName",
    "SpatialElementaryDistrictName"
]

for name, data in [("Training", train_fe), ("Test", test_fe)]:
    print(f"\n{name} dataset:")
    print("Property records:", len(data))
    print("Missing spatial district names:")
    print(data[district_columns].isna().sum())
    display(data[district_columns].head())


Training dataset:
Property records: 278710
Missing spatial district names:
SpatialHighDistrictName          200
SpatialElementaryDistrictName     29
dtype: int64


,SpatialHighDistrictName,SpatialElementaryDistrictName
0,Sweetwater Union High,National Elementary
1,Sweetwater Union High,National Elementary
2,San Luis Coastal Unified,San Luis Coastal Unified
3,Fort Bragg Unified,Fort Bragg Unified
4,East Side Union High,Evergreen Elementary



Test dataset:
Property records: 11684
Missing spatial district names:
SpatialHighDistrictName          11
SpatialElementaryDistrictName     0
dtype: int64


,SpatialHighDistrictName,SpatialElementaryDistrictName
0,San Jose Unified,San Jose Unified
1,Ventura Unified,Ventura Unified
2,Los Gatos-Saratoga Union High,Saratoga Union Elementary
3,Alameda Unified,Alameda Unified
4,Capistrano Unified,Capistrano Unified


In [33]:
# Label unmatched districts as a separate category
district_features = [
    "SpatialHighDistrictName",
    "SpatialElementaryDistrictName"
]

for name, data in [("Training", train_fe), ("Test", test_fe)]:
    data[district_features] = (
        data[district_features].fillna("Unmatched")
    )

    print(f"\n{name}: unmatched district counts")
    print(data[district_features].eq("Unmatched").sum())


Training: unmatched district counts
SpatialHighDistrictName          200
SpatialElementaryDistrictName     29
dtype: int64

Test: unmatched district counts
SpatialHighDistrictName          11
SpatialElementaryDistrictName     0
dtype: int64


In [34]:
# Exclude targets, sale dates, and district metadata
excluded_columns = [
    "ClosePrice",
    "CloseDate",
    "SpatialHighDistrictCode",
    "SpatialHighDistrictType",
    "SpatialElementaryDistrictCode",
    "SpatialElementaryDistrictType",
]

X_train_fe = train_fe.drop(
    columns=excluded_columns, errors="ignore"
).copy()

X_test_fe = test_fe.drop(
    columns=excluded_columns, errors="ignore"
).copy()

# Identify categorical and numeric features
categorical_columns_fe = X_train_fe.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

numeric_columns_fe = [
    col for col in X_train_fe.columns
    if col not in categorical_columns_fe
]

assert X_train_fe.columns.equals(X_test_fe.columns)

print("Training features:", X_train_fe.shape)
print("Test features:", X_test_fe.shape)

print("\nNumeric features:")
print(numeric_columns_fe)

print("\nCategorical features:")
print(categorical_columns_fe)

Training features: (278710, 29)
Test features: (11684, 29)

Numeric features:
['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'Stories', 'MainLevelBedrooms', 'GarageSpaces', 'LotSizeSquareFeet', 'PropertyAge', 'BedBathRatio', 'ZeroBathrooms']

Categorical features:
['Flooring', 'ViewYN', 'PoolPrivateYN', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'SubdivisionName', 'City', 'FireplaceYN', 'Levels', 'NewConstructionYN', 'HighSchoolDistrict', 'PostalCode', 'SpatialHighDistrictName', 'SpatialElementaryDistrictName']


In [38]:
# Use the same training data until April,2026 as test data
y_train_fe = pd.to_numeric(
    train_fe["ClosePrice"], errors="raise"
)

y_test_fe = pd.to_numeric(
    test_fe["ClosePrice"], errors="raise"
)

# Verify feature-target alignment
assert X_train_fe.index.equals(y_train_fe.index)
assert X_test_fe.index.equals(y_test_fe.index)
assert X_train_fe.columns.equals(X_test_fe.columns)

print("Training features:", X_train_fe.shape)
print("Test features:", X_test_fe.shape)
print("Training targets:", len(y_train_fe))
print("Test targets:", len(y_test_fe))

Training features: (278710, 29)
Test features: (11684, 29)
Training targets: 278710
Test targets: 11684


## Restrain linear regression model

In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Restore undefined ratios before fitting the imputer
for data in [X_train_fe, X_test_fe]:
    data.loc[
        data["ZeroBathrooms"].eq(1), "BedBathRatio"
    ] = float("nan")

# Standardize categorical labels
for data in [X_train_fe, X_test_fe]:
    for col in categorical_columns_fe:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
            .astype(str)
        )

numeric_pipeline_fe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_fe = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns_fe,
        ),
        ("numeric", numeric_pipeline_fe, numeric_columns_fe),
    ],
    sparse_threshold=1.0,
)

# Fit preprocessing on the full training set only
X_train_ready_fe = preprocessor_fe.fit_transform(X_train_fe)
X_test_ready_fe = preprocessor_fe.transform(X_test_fe)

print("Training matrix:", X_train_ready_fe.shape)
print("Test matrix:", X_test_ready_fe.shape)

Training matrix: (278710, 17751)
Test matrix: (11684, 17751)


In [40]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

# Train on the full training set, including March
linear_fe = LinearRegression()

print("Training Linear Regression...", flush=True)
linear_fe.fit(X_train_ready_fe, np.log(y_train_fe))

# Evaluate training and test performance
linear_rows_fe = []

for split, features, actual in [
    ("Training", X_train_ready_fe, y_train_fe),
    ("Test", X_test_ready_fe, y_test_fe),
]:
    log_predictions = linear_fe.predict(features)
    predictions = np.exp(log_predictions)

    linear_rows_fe.append({
        "Split": split,
        "R² (price)": r2_score(actual, predictions),
        "R² (log price)": r2_score(
            np.log(actual), log_predictions
        ),
        "MAE ($)": mean_absolute_error(actual, predictions),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predictions)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(actual, predictions)
            * 100
        ),
    })

linear_results_fe = pd.DataFrame(linear_rows_fe)

display(
    linear_results_fe.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Training Linear Regression...


Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
Training,0.3820,0.9313,"151,026.66","1,180,725.07",11.48
Test,0.8281,0.9268,"171,314.43","452,408.10",12.37


### Old vs. New Feature Sets

The log-target linear regression baseline uses 24 original features.
The updated model adds property age, bedroom-to-bathroom ratio,
a zero-bathroom indicator, and two spatial school district features.

The table compares their performance on the April 2026 test set.
Both models use the full pre-April training period, including March.

In [42]:
%pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 10.7 MB/s  0:00:00a 0:00:010:00:01

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
import json
from pathlib import Path
from io import StringIO
import pandas as pd

# Read saved outputs from Notebook 03 without rerunning it
baseline_path = Path("03_baseline_model.ipynb")

with baseline_path.open(encoding="utf-8") as file:
    baseline_notebook = json.load(file)

baseline_table = None

for cell in baseline_notebook["cells"]:
    for output in cell.get("outputs", []):
        html = output.get("data", {}).get("text/html", "")
        html = "".join(html) if isinstance(html, list) else html

        if "Log model — Test metrics" in html:
            baseline_table = pd.read_html(StringIO(html))[0]
            break

    if baseline_table is not None:
        break

if baseline_table is None:
    raise ValueError(
        "Saved log-model Test results not found. "
        "Save Notebook 03 with its metric outputs first."
    )

# Convert the saved metric values into numbers
old_metrics = dict(
    zip(
        baseline_table.iloc[:, 0],
        baseline_table.iloc[:, 1],
    )
)

def parse_metric(value):
    return float(
        str(value)
        .replace("$", "")
        .replace(",", "")
        .replace("%", "")
        .strip()
    )

# Get Test results from the current feature-engineered model
new_metrics = (
    linear_results_fe.set_index("Split").loc["Test"]
)

# Compare feature sets using Test metrics
feature_comparison = pd.DataFrame([
    {
        "Feature set": "Original features",
        "Features": 24,
        "School district layer": "No",
        "R² (price)": parse_metric(old_metrics["R² (price)"]),
        "MAE ($)": parse_metric(old_metrics["MAE"]),
        "RMSE ($)": parse_metric(old_metrics["RMSE"]),
        "MAPE (%)": parse_metric(old_metrics["MAPE"]),
    },
    {
        "Feature set": "Original + engineered features",
        "Features": X_train_fe.shape[1],
        "School district layer": "High + Elementary",
        "R² (price)": new_metrics["R² (price)"],
        "MAE ($)": new_metrics["MAE ($)"],
        "RMSE ($)": new_metrics["RMSE ($)"],
        "MAPE (%)": new_metrics["MAPE (%)"],
    },
])

display(
    feature_comparison.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Feature set,Features,School district layer,R² (price),MAE ($),RMSE ($),MAPE (%)
Original features,24,No,0.8256,"172,630.73","455,641.17",12.45
Original + engineered features,29,High + Elementary,0.8281,"171,314.43","452,408.10",12.37


The engineered feature set produced a modest improvement over the
original baseline on the April 2026 test set. Price-scale R² increased
from 0.8256 to 0.8281, while MAE, RMSE, and MAPE all decreased.

These results suggest that the five added features, including the
school district layer, provide some additional predictive information.
The comparison evaluates their combined contribution rather than
isolating the effect of the school district features.

## Retrain Decision Tree and Random Forest

In [48]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

# Use the depth selected in Notebook 04
dt_depth_fe = 20

models_fe = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=dt_depth_fe,
        min_samples_leaf=5,
        random_state=42,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.3,
        random_state=42,
        n_jobs=-1,
    ),
}

tree_results_rows_fe = []

for model_name, model in models_fe.items():
    print(f"Training {model_name}...", flush=True)

    # Fit on the full training set using log sale prices
    model.fit(X_train_ready_fe, np.log(y_train_fe))

    for split, features, actual in [
        ("Training", X_train_ready_fe, y_train_fe),
        ("Test", X_test_ready_fe, y_test_fe),
    ]:
        log_predictions = model.predict(features)
        predictions = np.exp(log_predictions)

        tree_results_rows_fe.append({
            "Model": model_name,
            "Split": split,
            "R² (price)": r2_score(actual, predictions),
            "R² (log price)": r2_score(
                np.log(actual), log_predictions
            ),
            "MAE ($)": mean_absolute_error(actual, predictions),
            "RMSE ($)": np.sqrt(
                mean_squared_error(actual, predictions)
            ),
            "MAPE (%)": (
                mean_absolute_percentage_error(actual, predictions)
                * 100
            ),
        })

    print(f"{model_name} completed.", flush=True)

tree_results_fe = pd.DataFrame(tree_results_rows_fe)

display(
    tree_results_fe.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Training Decision Tree...
Decision Tree completed.
Training Random Forest...
Random Forest completed.


Model,Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
Decision Tree,Training,0.4033,0.9453,"135,975.24","1,160,199.47",10.01
Decision Tree,Test,0.7479,0.8953,"204,703.81","547,840.88",14.38
Random Forest,Training,0.3824,0.9303,"153,763.70","1,180,330.89",11.37
Random Forest,Test,0.7707,0.9094,"192,876.20","522,483.31",13.40


In [49]:
import json
from pathlib import Path
from io import StringIO
import pandas as pd

# Read saved results from Notebook 04
comparison_path = Path("04_model_comparison.ipynb")

with comparison_path.open(encoding="utf-8") as file:
    notebook_04 = json.load(file)


def read_saved_table(source_marker):
    for cell in notebook_04["cells"]:
        source = "".join(cell.get("source", []))

        if source_marker not in source:
            continue

        for output in cell.get("outputs", []):
            html = output.get("data", {}).get("text/html", "")
            html = "".join(html) if isinstance(html, list) else html

            if "<table" in html:
                return pd.read_html(StringIO(html))[0]

    raise ValueError(f"Saved table not found: {source_marker}")


# Extract original Random Forest Test metrics
rf_old_table = read_saved_table("rf_metrics_df =")
rf_old = dict(zip(
    rf_old_table.iloc[:, 0],
    rf_old_table.iloc[:, 1],
))

# Extract original Decision Tree Test metrics
dt_old_table = read_saved_table("dt_metrics_df =")
dt_old = dt_old_table.set_index("Split").loc["Test"]


def parse_value(value):
    return float(
        str(value)
        .replace("$", "")
        .replace(",", "")
        .replace("%", "")
        .strip()
    )


old_tree_metrics = {
    "Decision Tree": {
        "R² (price)": parse_value(dt_old["R² (price)"]),
        "MAE ($)": parse_value(dt_old["MAE ($)"]),
        "RMSE ($)": parse_value(dt_old["RMSE ($)"]),
        "MAPE (%)": parse_value(dt_old["MAPE (%)"]),
    },
    "Random Forest": {
        "R² (price)": parse_value(rf_old["R² (price)"]),
        "MAE ($)": parse_value(rf_old["MAE"]),
        "RMSE ($)": parse_value(rf_old["RMSE"]),
        "MAPE (%)": parse_value(rf_old["MAPE"]),
    },
}

# Compare old and new feature sets on the Test set
comparison_rows = []
metric_columns = ["R² (price)", "MAE ($)", "RMSE ($)", "MAPE (%)"]

for model_name, old_metrics in old_tree_metrics.items():
    new_metrics = tree_results_fe.loc[
        tree_results_fe["Model"].eq(model_name)
        & tree_results_fe["Split"].eq("Test")
    ].iloc[0]

    comparison_rows.append({
        "Model": model_name,
        "Feature set": "Original features",
        "Features": 24,
        "School district layer": "No",
        **old_metrics,
    })

    comparison_rows.append({
        "Model": model_name,
        "Feature set": "Original + engineered features",
        "Features": X_train_fe.shape[1],
        "School district layer": "High + Elementary",
        **{col: new_metrics[col] for col in metric_columns},
    })

tree_feature_comparison = pd.DataFrame(comparison_rows)

display(
    tree_feature_comparison.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Model,Feature set,Features,School district layer,R² (price),MAE ($),RMSE ($),MAPE (%)
Decision Tree,Original features,24,No,0.7560,"200,844.60","538,983.26",14.32
Decision Tree,Original + engineered features,29,High + Elementary,0.7479,"204,703.81","547,840.88",14.38
Random Forest,Original features,24,No,0.7708,"190,535.11","522,371.80",13.21
Random Forest,Original + engineered features,29,High + Elementary,0.7707,"192,876.20","522,483.31",13.40


### Feature Engineering Results

With model hyperparameters held constant, adding the five engineered
features, including the school district layer, did not improve the
tree-based models on the April 2026 test set.

Decision Tree R² decreased from 0.7560 to 0.7479, with higher MAE,
RMSE, and MAPE. Random Forest R² remained nearly unchanged
(0.7708 versus 0.7707), while its error metrics increased slightly.

In contrast, log-target Linear Regression showed modest improvements
across all four test metrics. The benefit of the engineered feature
set therefore varied by model. These comparisons assess the five
features together and do not isolate the school district layer's effect.